# overpy — offline smoke test

In [ ]:
# Example 1: Parse a canned Overpass JSON payload into a Result (nodes + tags).
import overpy

payload = {
    "version": 0.6, "generator": "test", "osm3s": {},
    "elements": [
        {"type": "node", "id": 1, "lat": 37.7908, "lon": -122.4025,
         "tags": {"amenity": "cafe", "name": "Peet's Coffee"}},
        {"type": "node", "id": 2, "lat": 37.7914, "lon": -122.3994,
         "tags": {"amenity": "cafe", "name": "Flywheel Coffee"}},
        {"type": "node", "id": 3, "lat": 37.7967, "lon": -122.3998},
    ],
}
result = overpy.Result.from_json(payload)

print(f"nodes = {len(result.nodes)}, ways = {len(result.ways)}, relations = {len(result.relations)}")
for node in result.nodes:
    print(f"  {node.tags.get('name', 'Unnamed')} at ({node.lat:.4f}, {node.lon:.4f})")
assert len(result.nodes) == 3
assert result.get_node(1).tags["name"] == "Peet's Coffee"

In [ ]:
# Example 2: Parse an Overpass XML payload (the second wire format overpy speaks).
import overpy
from decimal import Decimal

xml = '''<?xml version="1.0" encoding="UTF-8"?>
<osm version="0.6" generator="test">
  <node id="100" lat="51.2345" lon="7.1234"><tag k="name" v="Fountain"/></node>
  <node id="101" lat="51.2350" lon="7.1240"/>
  <way id="200"><nd ref="100"/><nd ref="101"/><tag k="highway" v="path"/></way>
</osm>'''

result = overpy.Overpass().parse_xml(xml)
print(f"nodes = {len(result.nodes)}, ways = {len(result.ways)}")

fountain = result.get_node(100)
print(f"node 100: tags={fountain.tags}, lat={fountain.lat} ({type(fountain.lat).__name__})")
assert fountain.tags == {"name": "Fountain"}
assert isinstance(fountain.lat, Decimal)  # XML parse path yields exact Decimal coordinates
assert result.ways[0].tags["highway"] == "path"

In [ ]:
# Example 3: Way → member-node resolution from the same result set.
import overpy

payload = {
    "version": 0.6, "generator": "test", "osm3s": {},
    "elements": [
        {"type": "node", "id": 1, "lat": 51.0, "lon": 7.0},
        {"type": "node", "id": 2, "lat": 51.1, "lon": 7.1},
        {"type": "node", "id": 3, "lat": 51.2, "lon": 7.2},
        {"type": "way", "id": 10, "nodes": [1, 2, 3],
         "tags": {"highway": "residential", "name": "Test St"}},
    ],
}
result = overpy.Result.from_json(payload)

way = result.get_way(10)
member_nodes = way.get_nodes(resolve_missing=False)
print(f"way {way.id} ({way.tags['name']}): node ids = {[n.id for n in member_nodes]}")
assert [n.id for n in member_nodes] == [1, 2, 3]
assert way.tags["highway"] == "residential"

In [ ]:
# Example 4: Relations — typed members (RelationNode / RelationWay) with roles.
import overpy

payload = {
    "version": 0.6, "generator": "test", "osm3s": {},
    "elements": [
        {"type": "node", "id": 1, "lat": 51.0, "lon": 7.0, "tags": {"name": "Trailhead"}},
        {"type": "node", "id": 2, "lat": 51.1, "lon": 7.1},
        {"type": "way", "id": 10, "nodes": [1, 2], "tags": {"highway": "path"}},
        {"type": "relation", "id": 100,
         "tags": {"type": "route", "route": "hiking", "name": "Ridge Trail"},
         "members": [
             {"type": "node", "ref": 1, "role": "stop"},
             {"type": "way", "ref": 10, "role": ""},
         ]},
    ],
}
result = overpy.Result.from_json(payload)

rel = result.get_relation(100)
members = [(type(m).__name__, m.ref, m.role) for m in rel.members]
print(f"relation {rel.id} ({rel.tags['name']}): members = {members}")
assert members == [("RelationNode", 1, "stop"), ("RelationWay", 10, "")]
assert rel.tags["route"] == "hiking"

In [ ]:
# Example 5: GeoJSON export round-trip from a parsed Result.
import json

import overpy

payload = {
    "version": 0.6, "generator": "test", "osm3s": {},
    "elements": [
        {"type": "node", "id": i, "lat": 37.77 + i * 0.001, "lon": -122.45 + i * 0.001,
         "tags": {"amenity": "library", "name": f"Branch {i}"}}
        for i in range(1, 6)
    ],
}
result = overpy.Result.from_json(payload)

geojson = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature",
         "geometry": {"type": "Point", "coordinates": [float(n.lon), float(n.lat)]},
         "properties": dict(n.tags)}
        for n in result.nodes
    ],
}
round_tripped = json.loads(json.dumps(geojson))
print(f"exported {len(round_tripped['features'])} features; "
      f"first = {round_tripped['features'][0]['properties']['name']}")
assert len(round_tripped["features"]) == 5
assert round_tripped["features"][0]["geometry"]["type"] == "Point"